In [9]:
!pip install pandas_ta
!pip install yfinance

In [27]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np

In [28]:
ticker_list = ["AAPL", "MSFT", "GOOGL", "NVDA", "META"]

In [29]:
df_bulk = yf.download(tickers=ticker_list, start="2015-01-01", end="2023-12-31", group_by="ticker")

/tmp/ipykernel_50582/3797272726.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_bulk = yf.download(tickers=ticker_list, start="2015-01-01", end="2023-12-31", group_by="ticker")
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[                       0%                       ]/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/yfina

In [30]:
df_stacked = df_bulk.stack(level=0, future_stack=True).rename_axis(['Date', 'Ticker']).reset_index()
df_stacked = df_stacked.sort_values(by=['Date', 'Ticker'])
df_stacked.columns.name = None
df_stacked = df_stacked.set_index(['Date', 'Ticker'])
print(df_stacked.head(10))

                        Open       High        Low      Close     Volume
Date       Ticker                                                       
2015-01-02 AAPL    24.671153  24.682228  23.776355  24.214895  212818400
           GOOGL   26.430299  26.589101  26.196068  26.278944   26480000
           META    78.034895  78.382466  77.160995  77.905792   18177500
           MSFT    39.682644  40.328995  39.580589  39.767689   27913900
           NVDA     0.483011   0.486611   0.475333   0.483011  113680000
2015-01-05 AAPL    23.984549  24.064284  23.346674  23.532721  257142000
           GOOGL   26.159844  26.201529  25.693369  25.778227   41182000
           META    77.439077  78.700264  76.326844  76.654556   26452200
           MSFT    39.436009  39.742176  39.333954  39.401993   39673900
           NVDA     0.483011   0.484451   0.472694   0.474853  197952000


In [31]:
def calculate_features(group):
    # Group 1: pandas-ta features
    group.ta.rsi(length=14, append=True)
    group.ta.ppo(append=True)
    group.ta.obv(append=True)
    group.ta.atr(length=14, append=True)

    # Group 2: Custom Pandas Math
    group['Log_Return'] = np.log(group['Close'] / group['Close'].shift(1))
    group['Overnight_Gap'] = (group['Open'] - group['Close'].shift(1)) / group['Close'].shift(1)
    group['High_Low_Ratio'] = (group['High'] - group['Low']) / group['Close']

    return group

In [32]:
df_features = df_stacked.groupby('Ticker', group_keys=False).apply(calculate_features)
df_features = df_features.dropna()
df_features.head(10)

Open       High        Low      Close     Volume  \
Date       Ticker                                                          
2015-02-09 AAPL    26.360607  26.647448  26.333923  26.620766  155559200   
           GOOGL   26.353876  26.493820  26.179693  26.265545   30314000   
           META    73.536328  74.310916  72.940484  73.923622   16194300   
           MSFT    35.923584  36.348816  35.898068  36.025639   31381100   
           NVDA     0.486611   0.490450   0.483491   0.489010  273944000   
2015-02-10 AAPL    26.720826  27.161096  26.718603  27.132189  248034000   
           GOOGL   26.407970  26.847151  26.260087  26.805466   47430000   
           META    74.330766  74.817365  73.983195  74.668411   15811300   
           MSFT    36.348827  36.374340  35.872566  36.229759   29670700   
           NVDA     0.489490   0.502927   0.487330   0.502687  226204000   

                      RSI_14  PPO_12_26_9  PPOh_12_26_9  PPOs_12_26_9  \
Date       Ticker                                                       
2015-02-09 AAPL    49.439620     3.792266      0.000000      3.792266   
           GOOGL   41.485745     1.853540      0.000000      1.853540   
           META    29.966050    -0.473728      0.000000     -0.473728   
           MSFT    34.843326    -4.477788      0.000000     -4.477788   
           NVDA    45.499335     0.542067      0.000000      0.542067   
2015-02-10 AAPL    53.791662     4.028736      0.189176      3.839560   
           GOOGL   47.585383     1.744873     -0.086934      1.831807   
           META    34.365419    -0.598643     -0.099932     -0.498711   
           MSFT    36.587299    -4.991546     -0.411006     -4.580539   
           NVDA    52.587252     0.483399     -0.046934      0.530333   

                            OBV   ATRr_14  Log_Return  Overnight_Gap  \
Date       Ticker                                                      
2015-02-09 AAPL    2.051296e+09  0.639403    0.006621      -0.003195   
           GOOGL   1.231140e+08  0.573254   -0.008654      -0.005282   
           META   -6.410260e+07  1.730914   -0.000403      -0.005640   
           MSFT   -2.638299e+08  0.978625   -0.001180      -0.004009   
           NVDA   -4.848080e+08  0.012295   -0.000981      -0.005882   
2015-02-10 AAPL    2.299330e+09  0.632326    0.019029       0.003759   
           GOOGL   1.705440e+08  0.574240    0.020348       0.005422   
           META   -4.829130e+07  1.671116    0.010025       0.005508   
           MSFT   -2.341592e+08  0.944564    0.005650       0.008971   
           NVDA   -2.586040e+08  0.012531    0.027585       0.000981   

                   High_Low_Ratio  
Date       Ticker                  
2015-02-09 AAPL          0.011777  
           GOOGL         0.011960  
           META          0.018538  
           MSFT          0.012512  
           NVDA          0.014230  
2015-02-10 AAPL          0.016309  
           GOOGL         0.021901  
           META          0.011172  
           MSFT          0.013850  
           NVDA          0.031026

In [33]:
# ^TNX = 10-Year Treasury Yield
# ^IRX = 13-Week Treasury Bill Yield (Often used as the short-term proxy)
# ^VIX = Cboe Volatility Index
# DX-Y.NYB = US Dollar Index
# XLK = Tech Sector ETF
benchmark_tickers = ['^TNX', '^IRX', '^VIX', 'DX-Y.NYB', 'XLK']

df_bench = yf.download(benchmark_tickers, start="2015-01-01", end="2023-12-31")['Close']
df_macro = pd.DataFrame(index=df_bench.index)
df_macro['Term_Spread'] = df_bench['^TNX'] - df_bench['^IRX']
df_macro['VIX'] = df_bench['^VIX']
df_macro['USD_Index'] = df_bench['DX-Y.NYB']
df_macro['XLK_Log_Return'] = np.log(df_bench['XLK'] / df_bench['XLK'].shift(1))
df_macro = df_macro.dropna().reset_index()
if 'Ticker' not in df_features.columns:
    df_features = df_features.reset_index()
df_final = pd.merge(df_features, df_macro, on='Date', how='left')
df_final['Sector_Relative_Strength'] = df_final['Log_Return'] - df_final['XLK_Log_Return']
df_final = df_final.drop(columns=['XLK_Log_Return'])

/tmp/ipykernel_50582/3213303487.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_bench = yf.download(benchmark_tickers, start="2015-01-01", end="2023-12-31")['Close']
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[                       0%                       ]/usr/local/lib/python3.12/dist-packages/yfinance/scraper

In [35]:
df_final.head(10)

,Date,Ticker,Open,High,Low,Close,Volume,RSI_14,PPO_12_26_9,PPOh_12_26_9,PPOs_12_26_9,OBV,ATRr_14,Log_Return,Overnight_Gap,High_Low_Ratio,Term_Spread,VIX,USD_Index,Sector_Relative_Strength
0,2015-02-09,AAPL,26.360607,26.647448,26.333923,26.620766,155559200,49.439620,3.792266,0.000000,3.792266,2.051296e+09,0.639403,0.006621,-0.003195,0.011777,1.945,18.549999,94.449997,0.008081
1,2015-02-09,GOOGL,26.353876,26.493820,26.179693,26.265545,30314000,41.485745,1.853540,0.000000,1.853540,1.231140e+08,0.573254,-0.008654,-0.005282,0.011960,1.945,18.549999,94.449997,-0.007193
2,2015-02-09,META,73.536328,74.310916,72.940484,73.923622,16194300,29.966050,-0.473728,0.000000,-0.473728,-6.410260e+07,1.730914,-0.000403,-0.005640,0.018538,1.945,18.549999,94.449997,0.001058
3,2015-02-09,MSFT,35.923584,36.348816,35.898068,36.025639,31381100,34.843326,-4.477788,0.000000,-4.477788,-2.638299e+08,0.978625,-0.001180,-0.004009,0.012512,1.945,18.549999,94.449997,0.000281
4,2015-02-09,NVDA,0.486611,0.490450,0.483491,0.489010,273944000,45.499335,0.542067,0.000000,0.542067,-4.848080e+08,0.012295,-0.000981,-0.005882,0.014230,1.945,18.549999,94.449997,0.000480
5,2015-02-10,AAPL,26.720826,27.161096,26.718603,27.132189,248034000,53.791662,4.028736,0.189176,3.839560,2.299330e+09,0.632326,0.019029,0.003759,0.016309,1.986,17.230000,94.760002,0.004275
6,2015-02-10,GOOGL,26.407970,26.847151,26.260087,26.805466,47430000,47.585383,1.744873,-0.086934,1.831807,1.705440e+08,0.574240,0.020348,0.005422,0.021901,1.986,17.230000,94.760002,0.005594
7,2015-02-10,META,74.330766,74.817365,73.983195,74.668411,15811300,34.365419,-0.598643,-0.099932,-0.498711,-4.829130e+07,1.671116,0.010025,0.005508,0.011172,1.986,17.230000,94.760002,-0.004730
8,2015-02-10,MSFT,36.348827,36.374340,35.872566,36.229759,29670700,36.587299,-4.991546,-0.411006,-4.580539,-2.341592e+08,0.944564,0.005650,0.008971,0.013850,1.986,17.230000,94.760002,-0.009104
9,2015-02-10,NVDA,0.489490,0.502927,0.487330,0.502687,226204000,52.587252,0.483399,-0.046934,0.530333,-2.586040e+08,0.012531,0.027585,0.000981,0.031026,1.986,17.230000,94.760002,0.012830
